In [1]:
from agents.suits import SUITS, SUIT_COLORS, DECKS, Cards, Market
from agents.traders.hand_reading import HandTracker, StartingHand, FundamentalistExpectedValue, StartingHandFilter
from agents.traders.hand_reading import HandReading

import math

starting_hand_size = 10
STARTING_HAND_INDEX = []
# Create the starting hand index structure
for i in range(starting_hand_size+1):
    for j in range(starting_hand_size-i+1):
        for k in range(starting_hand_size-i-j+1):
            l = starting_hand_size-i-j-k
            STARTING_HAND_INDEX.append(Cards(i, j, k, l))

In [2]:
def update_deck_probabilities(starting_hand, oppo_starting_hands, alpha) -> None:
    """
    Update the probabilities that each deck is the one in play.
    Uses the expected values of cards that you think each player has in their hand.
    """
    expected_total = starting_hand.copy()
    print(expected_total)

    # Accumulate expected counts
    for starting_hand_dist in oppo_starting_hands.values():
        expected_total += starting_hand_dist.expected_value()

    print(expected_total)

    # Compute squared error loss for each possible deck
    total_loss = [
        sum((expected_total[suit] - deck[suit])**2 for suit in SUITS) / len(SUITS)
        for deck in DECKS
    ]
    weights = [ math.exp(-alpha * loss) for loss in total_loss ]
    
    # Normalize weights and return dictionary mapping between deck and weight
    total = sum(weights)
    return [weight/total for weight in weights]

In [3]:
def multinomial_deck_probability(known_cards: Cards[int]) -> list[float]:
    """
    Compute normalized multinomial probability weight.
    Probability a given deck is the current deck in play given the known cards.
    """
    m_values = []
    # Generate all possible decks
    for deck in DECKS:
            m = 1
            for suit in SUITS:
                k = known_cards[suit]
                n = deck[suit]
                if k > n:
                    m = 0
                    break
                m *= math.comb(n, k)
            m_values.append(m)

    total = sum(m_values)
    # All decks are impossible
    if total == 0:
        return [0 * len(m_values)]
    return [m / total for m in m_values]

In [4]:
opponents = {"opp1", "opp2", "opp3"}

oppo_starting_hands = {pid: StartingHand(evidence_sensitivity=0.5, buy_ratio=1.7, hand_index=STARTING_HAND_INDEX) for pid in opponents}
initials = {player: Cards() for player in opponents | {"me"}}

hand = Cards(1, 3, 4, 2)
initials["opp1"] = Cards(1, 0, 2, 0)
initials["opp2"] = Cards(0, 3, 1, 0)
initials["opp3"] = Cards(3, 0, 1, 2)
initials["me"] =   Cards(1, 2, 0, 0)

observed = dict()
for player in initials:
    total = Cards()
    for p, cards in initials.items():
        if p != player:
            total += cards
    observed[player] = total

known_cards = hand + initials["opp1"] + initials["opp2"] + initials["opp3"]

for pid in oppo_starting_hands:
    oppo_starting_hands[pid].set_base(initials[pid], 
                                      hand + observed[pid])

In [5]:
print(hand)

mult_deck_probs = multinomial_deck_probability(known_cards)

for deck, prob in zip(DECKS, mult_deck_probs):
  print(deck, prob)

♥1 ♠3 ♣4 ♦2
♥10 ♠10 ♣8 ♦12 0.004174784302811021
♥10 ♠10 ♣12 ♦8 0.2922349011967715
♥8 ♠10 ♣10 ♦12 0.041747843028110215
♥10 ♠12 ♣8 ♦10 0.007792930698580574
♥10 ♠8 ♣10 ♦12 0.025048705816866128
♥12 ♠8 ♣10 ♦10 0.03339827442248817
♥12 ♠10 ♣8 ♦10 0.0055663790704146955
♥8 ♠10 ♣12 ♦10 0.19482326746451434
♥8 ♠12 ♣10 ♦10 0.07792930698580573
♥10 ♠8 ♣12 ♦10 0.1168939604787086
♥12 ♠10 ♣10 ♦8 0.08349568605622043
♥10 ♠12 ♣10 ♦8 0.1168939604787086


In [6]:
calc_deck_probs = update_deck_probabilities(hand, oppo_starting_hands, .60)

for deck, prob in zip(DECKS, calc_deck_probs):
  print(deck, prob)

♥1 ♠3 ♣4 ♦2
♥9.386366983061377 ♠9.873935447578603 ♣12.0351012272851 ♦8.704596342074922
♥10 ♠10 ♣8 ♦12 0.005814383305177521
♥10 ♠10 ♣12 ♦8 0.3163789139985404
♥8 ♠10 ♣10 ♦12 0.02849063037531946
♥10 ♠12 ♣8 ♦10 0.01172751970231476
♥10 ♠8 ♣10 ♦12 0.021264397508245943
♥12 ♠8 ♣10 ♦10 0.03201154376302885
♥12 ♠10 ♣8 ♦10 0.008753005372314826
♥8 ♠10 ♣12 ♦10 0.21016190410229585
♥8 ♠12 ♣10 ♦10 0.0574651534859078
♥10 ♠8 ♣12 ♦10 0.15685740227750128
♥12 ♠10 ♣10 ♦8 0.06456678042676274
♥10 ♠12 ♣10 ♦8 0.0865083656825906


In [7]:
oppo_starting_hands["opp1"].add_filter('bid', 'spades', 8, observed["opp1"], Cards())

In [8]:
calc_deck_probs = update_deck_probabilities(hand, oppo_starting_hands, .60)

for deck, prob in zip(DECKS, calc_deck_probs):
  print(deck, prob)

♥1 ♠3 ♣4 ♦2
♥9.017567735405164 ♠9.816377420490497 ♣12.68797472520029 ♦8.47808011890405
♥10 ♠10 ♣8 ♦12 0.0023158036464128577
♥10 ♠10 ♣12 ♦8 0.36199496479772947
♥8 ♠10 ♣10 ♦12 0.020947115850394615
♥10 ♠12 ♣8 ♦10 0.005169289004351538
♥10 ♠8 ♣10 ♦12 0.012970987823380221
♥12 ♠8 ♣10 ♦10 0.017928788688047943
♥12 ♠10 ♣8 ♦10 0.0032009554541950905
♥8 ♠10 ♣12 ♦10 0.2618934475839109
♥8 ♠12 ♣10 ♦10 0.046757718775531476
♥10 ♠8 ♣12 ♦10 0.1621710952426885
♥12 ♠10 ♣10 ♦8 0.04002027130841898
♥10 ♠12 ♣10 ♦8 0.06462956182493845


In [12]:
for hand, filter_val in zip(STARTING_HAND_INDEX, oppo_starting_hands["opp1"].computed_filters['spades']['bid']):
  print(hand, "val=", filter_val)

♥0 ♠0 ♣0 ♦10 val= 0
♥0 ♠0 ♣1 ♦9 val= 0
♥0 ♠0 ♣2 ♦8 val= 0
♥0 ♠0 ♣3 ♦7 val= 0
♥0 ♠0 ♣4 ♦6 val= 0
♥0 ♠0 ♣5 ♦5 val= 0
♥0 ♠0 ♣6 ♦4 val= 0
♥0 ♠0 ♣7 ♦3 val= 0
♥0 ♠0 ♣8 ♦2 val= 0
♥0 ♠0 ♣9 ♦1 val= 0
♥0 ♠0 ♣10 ♦0 val= 0
♥0 ♠1 ♣0 ♦9 val= 0
♥0 ♠1 ♣1 ♦8 val= 0
♥0 ♠1 ♣2 ♦7 val= 0
♥0 ♠1 ♣3 ♦6 val= 0
♥0 ♠1 ♣4 ♦5 val= 0
♥0 ♠1 ♣5 ♦4 val= 0
♥0 ♠1 ♣6 ♦3 val= 0
♥0 ♠1 ♣7 ♦2 val= 0
♥0 ♠1 ♣8 ♦1 val= 0
♥0 ♠1 ♣9 ♦0 val= 0
♥0 ♠2 ♣0 ♦8 val= 0
♥0 ♠2 ♣1 ♦7 val= 0
♥0 ♠2 ♣2 ♦6 val= 0
♥0 ♠2 ♣3 ♦5 val= 0
♥0 ♠2 ♣4 ♦4 val= 0
♥0 ♠2 ♣5 ♦3 val= 0
♥0 ♠2 ♣6 ♦2 val= 0
♥0 ♠2 ♣7 ♦1 val= 0
♥0 ♠2 ♣8 ♦0 val= 0
♥0 ♠3 ♣0 ♦7 val= 0
♥0 ♠3 ♣1 ♦6 val= 0
♥0 ♠3 ♣2 ♦5 val= 0
♥0 ♠3 ♣3 ♦4 val= 0
♥0 ♠3 ♣4 ♦3 val= 0
♥0 ♠3 ♣5 ♦2 val= 0
♥0 ♠3 ♣6 ♦1 val= 0
♥0 ♠3 ♣7 ♦0 val= 0
♥0 ♠4 ♣0 ♦6 val= 0
♥0 ♠4 ♣1 ♦5 val= 0
♥0 ♠4 ♣2 ♦4 val= 0
♥0 ♠4 ♣3 ♦3 val= 0
♥0 ♠4 ♣4 ♦2 val= 0
♥0 ♠4 ♣5 ♦1 val= 0
♥0 ♠4 ♣6 ♦0 val= 0
♥0 ♠5 ♣0 ♦5 val= 0
♥0 ♠5 ♣1 ♦4 val= 0
♥0 ♠5 ♣2 ♦3 val= 0
♥0 ♠5 ♣3 ♦2 val= 0
♥0 ♠5 ♣4 ♦1 val= 0
♥0 ♠5 ♣5 ♦0 val= 0
♥0 ♠6 ♣0 ♦4 val= 0
♥0 ♠6 ♣1 ♦

In [ ]:
player = HandReading("", "me", 0.5, test_mode=True)

hand = {"spades": 2,
        "clubs": 1,
        "hearts": 3,
        "diamonds": 4}
opponents = {"opp1", "opp2", "opp3"}

player._handle_start(hand, opponents)
player.update_deck_probability()

for deck, prob in zip(DECKS, player.deck_probabilities):
  print(deck, prob)

In [9]:
player._handle_offer("opp1", 4, "clubs")
player.update_deck_probability()

for deck, prob in zip(DECKS, player.deck_probabilities):
  print(deck, prob)

♥10 ♠10 ♣8 ♦12 0.11699687563408016
♥10 ♠10 ♣12 ♦8 0.056751422034865724
♥8 ♠10 ♣10 ♦12 0.09945827795616442
♥10 ♠12 ♣8 ♦10 0.09223499181552741
♥10 ♠8 ♣10 ♦12 0.10336037036858428
♥12 ♠8 ♣10 ♦10 0.08468151663579473
♥12 ♠10 ♣8 ♦10 0.09585368971698924
♥8 ♠10 ♣12 ♦10 0.06926951988467424
♥8 ♠12 ♣10 ♦10 0.07840836264691763
♥10 ♠8 ♣12 ♦10 0.07198720285192897
♥12 ♠10 ♣10 ♦8 0.06675903908970557
♥10 ♠12 ♣10 ♦8 0.0642387313647677


In [10]:
print(player.observed_cards("me"))

♥0 ♠0 ♣1 ♦0
